# NIRF time-series, trend, and performance-gap analysis

This notebook answers ten analysis questions using the shared `engine` package and `nirf_master_final.csv`. Analyses are filtered within a selected NIRF category, year range, and reporting year. The Plotly analysis functions return `(result_df, plotly_fig)` and can be called from a Streamlit app without notebook display code.

**Rank interpretation:** rank 1 is best. Exact ranks are the default for endpoint comparisons. The Question 2/3 checkbox includes `rank_value` midpoint estimates for rank bands, separates them from exact-rank movers, and shows rank-band uncertainty ranges; it does not convert bands into exact ranks. The rank-trend chart distinguishes band estimates with hollow markers and uncertainty ranges. Volatility is exact-rank-only by default; the optional estimate checkbox includes band midpoint estimates and labels them as estimates. Volatility also requires a minimum 60% of the selected years to be ranked so sparse histories are not mistaken for extreme movement. Questions 6, 7, and 9 use a single selected year to avoid repeating the same institution across years. Question 8 subtracts the same-year cohort median change before flagging score changes.

**Dashboard integration:** import `load_master`, `filter_data`, and any `question_*` function from `engine`. For example:

```python
results, fig = question_2_improvements(
    df, category=category, start_year=start_year, end_year=end_year,
    include_estimates=include_estimates,
)
st.plotly_chart(fig, use_container_width=True)
st.dataframe(results)
```

The interactive settings below are notebook defaults; a Streamlit sidebar can supply the same values through the `**filters` function arguments.

In [1]:
from pathlib import Path
import sys

from IPython.display import Markdown, display
import ipywidgets as widgets

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "engine").is_dir():
    raise FileNotFoundError(
        f"Open this notebook from the project root; shared engine package not found under {PROJECT_DIR}."
    )
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from engine import (
    endpoint_cohort_summary,
    filter_data,
    load_master,
    question_1_rank_trend,
    question_2_improvements,
    question_3_declines,
    question_4_score_trend,
    question_5_top10,
    question_6_research_vs_score,
    question_7_research_vs_teaching,
    question_8_adjusted_score_changes,
    question_9_parameter_vs_rank,
    question_10_rank_volatility,
)

# These values can be replaced by Streamlit sidebar controls in the app.
selected_category = "Overall"
selected_institution = "Indian Institute of Technology Madras"
selected_parameter = "RPC"  # One of TLR, RPC, GO, OI, PR
start_year = 2017
end_year = 2025
selected_year = 2025
include_rank_estimates = widgets.Checkbox(
    value=False,
    description="Include rank-band midpoint estimates in Questions 2 and 3",
    indent=False,
)
display(include_rank_estimates)
include_rank_estimates_for_volatility = False  # Exact ranks only by default in Question 10.
show_rank_band_estimates = True  # Show midpoint estimates and band ranges in Question 1.

CSV_PATH = PROJECT_DIR / "nirf_master_final.csv"
df = load_master(CSV_PATH)
if selected_institution not in set(
    df.loc[df["category"].eq(selected_category), "institute_name"]
):
    raise ValueError(
        f"{selected_institution!r} is not present in the selected category {selected_category!r}."
    )
if selected_parameter not in {"TLR", "RPC", "GO", "OI", "PR"}:
    raise ValueError("selected_parameter must be one of TLR, RPC, GO, OI, or PR.")

print(
    f"Loaded {len(df):,} validated rows from {CSV_PATH.name}; "
    f"category: {selected_category}; years: {int(df.year.min())}–{int(df.year.max())}."
)
print(
    f"Rank comparison: {start_year}–{end_year}; reporting year: {selected_year}; "
    f"include endpoint rank estimates: {include_rank_estimates.value}; "
    f"volatility band estimates: {include_rank_estimates_for_volatility}."
)

Checkbox(value=False, description='Include rank-band midpoint estimates in Questions 2 and 3', indent=False)

Loaded 11,012 validated rows from nirf_master_final.csv; category: Overall; years: 2017–2025.
Rank comparison: 2017–2025; reporting year: 2025; include endpoint rank estimates: False; volatility band estimates: False.


In [2]:
ANALYSIS_FUNCTIONS = [
    ("1", "Selected institution rank trend", question_1_rank_trend, {
        "category": selected_category,
        "institution": selected_institution,
        "start_year": start_year,
        "end_year": end_year,
        "include_estimates": show_rank_band_estimates,
    }),
    ("2", "Largest rank improvements", question_2_improvements, {
        "category": selected_category,
        "start_year": start_year,
        "end_year": end_year,
        "include_estimates": include_rank_estimates.value,
    }),
    ("3", "Largest rank declines", question_3_declines, {
        "category": selected_category,
        "start_year": start_year,
        "end_year": end_year,
        "include_estimates": include_rank_estimates.value,
    }),
    ("4", "Annual mean score and interquartile range", question_4_score_trend, {
        "category": selected_category,
        "start_year": start_year,
        "end_year": end_year,
    }),
    ("5", "Top-10 institution-by-year heatmap", question_5_top10, {
        "category": selected_category,
        "start_year": start_year,
        "end_year": end_year,
    }),
    ("6", "Research score versus overall score", question_6_research_vs_score, {
        "category": selected_category,
        "year": selected_year,
    }),
    ("7", "Research score versus teaching score", question_7_research_vs_teaching, {
        "category": selected_category,
        "year": selected_year,
    }),
    ("8", "Year-adjusted score changes", question_8_adjusted_score_changes, {
        "category": selected_category,
        "start_year": start_year,
        "end_year": end_year,
    }),
    ("9", "Selected parameter versus rank", question_9_parameter_vs_rank, {
        "category": selected_category,
        "year": selected_year,
        "parameter": selected_parameter,
    }),
    ("10", "Rank-volatility outliers and rank paths", question_10_rank_volatility, {
        "category": selected_category,
        "start_year": start_year,
        "end_year": end_year,
        "include_estimates": include_rank_estimates_for_volatility,
    }),
]

analysis_results = {}
analysis_figures = {}
for question_number, question_title, analysis_function, filters in ANALYSIS_FUNCTIONS:
    result_df, plotly_fig = analysis_function(df, **filters)
    analysis_results[question_number] = result_df
    analysis_figures[question_number] = plotly_fig
    display(Markdown(f"## Question {question_number}: {question_title}"))
    if result_df.empty:
        print("No matching rows for the current filters.")
    else:
        display(result_df.head(15).round(3))
    display(plotly_fig)

display(Markdown("## Endpoint coverage: exact ranks versus rank-band estimates"))
endpoint_coverage = endpoint_cohort_summary(
    df, category=selected_category, start_year=start_year, end_year=end_year
)
display(endpoint_coverage)

print(
    "Dashboard functions are available from engine.analysis. "
    "Each question_* function accepts df and keyword filters and returns (result_df, plotly_fig)."
)

## Question 1: Selected institution rank trend

,institution_key,institute_name,institute_id,institute_name_original,year,category,category_original,category_group,city,city_original,...,is_top100,is_new_entry,first_year_in_category,category_first_year,categories_in_year,source_page,weight_scheme_basis,has_score,has_components,score_comparison_group
501,INST0585,Indian Institute of Technology Madras,IR17-I-1-1-77,Indian Institute of Technology Madras,2017,Overall,Overall,General,Chennai,Chennai,...,True,False,2017,2017,6,OverallRanking,estimated_from_data,True,True,Overall | 30/30/20/10/10
1546,INST0585,Indian Institute of Technology Madras,IR-2-O-OE-U-0456,Indian Institute of Technology Madras,2018,Overall,Overall,General,Chennai,Chennai,...,True,False,2017,2017,9,OverallRanking,estimated_from_data,True,True,Overall | 30/30/20/10/10
2605,INST0585,Indian Institute of Technology Madras,IR-O-U-0456,Indian Institute of Technology Madras,2019,Overall,Overall,General,Chennai,Chennai,...,True,False,2017,2017,9,OverallRanking,estimated_from_data,True,True,Overall | 30/30/20/10/10
3814,INST0585,Indian Institute of Technology Madras,IR-O-U-0456,Indian Institute of Technology Madras,2020,Overall,Overall,General,Chennai,Chennai,...,True,False,2017,2017,10,OverallRanking,estimated_from_data,True,True,Overall | 30/30/20/10/10
5059,INST0585,Indian Institute of Technology Madras,IR-O-U-0456,Indian Institute of Technology Madras,2021,Overall,Overall,General,Chennai,Chennai,...,True,False,2017,2017,11,OverallRanking,estimated_from_data,True,True,Overall | 30/30/20/10/10
6384,INST0585,Indian Institute of Technology Madras,IR-O-U-0456,Indian Institute of Technology Madras,2022,Overall,Overall,General,Chennai,Chennai,...,True,False,2017,2017,11,OverallRanking,estimated_from_data,True,True,Overall | 30/30/20/10/10
7685,INST0585,Indian Institute of Technology Madras,IR-O-U-0456,Indian Institute of Technology Madras,2023,Overall,Overall,General,Chennai,Chennai,...,True,False,2017,2017,13,OverallRanking,estimated_from_data,True,True,Overall | 30/30/20/10/10
9008,INST0585,Indian Institute of Technology Madras,IR-O-U-0456,Indian Institute of Technology Madras,2024,Overall,Overall,General,Chennai,Chennai,...,True,False,2017,2017,16,OverallRanking,estimated_from_data,True,True,Overall | 30/30/20/10/10
10384,INST0585,Indian Institute of Technology Madras,IR-O-U-0456,Indian Institute of Technology Madras,2025,Overall,Overall,General,Chennai,Chennai,...,True,False,2017,2017,16,OverallRanking,estimated_from_data,True,True,Overall | 30/30/20/10/10


## Question 2: Largest rank improvements

year,institution_key,institute_name,rank_start,rank_end,rank_start_low,rank_start_high,rank_end_low,rank_end_high,start_year,end_year,rank_change,rank_change_low,rank_change_high,uses_estimated_rank
0,INST1276,Saveetha Institute of Medical and Technical Sc...,91.0,23.0,91.0,91.0,23.0,23.0,2017,2025,68.0,68.0,68.0,False
1,INST1480,Symbiosis International,97.0,40.0,97.0,97.0,40.0,40.0,2017,2025,57.0,57.0,57.0,False
2,INST0735,Kalinga Institute of Industrial Technology,79.0,27.0,79.0,79.0,27.0,27.0,2017,2025,52.0,52.0,52.0,False
3,INST0055,Amity University - Gautam Budh Nagar,86.0,37.0,86.0,86.0,37.0,37.0,2017,2025,49.0,49.0,49.0,False
4,INST0588,Indian Institute of Technology Patna,83.0,36.0,83.0,83.0,36.0,36.0,2017,2025,47.0,47.0,47.0,False
5,INST0569,Indian Institute of Technology (Banaras Hindu ...,70.0,31.0,70.0,70.0,31.0,31.0,2017,2025,39.0,39.0,39.0,False
6,INST0576,Indian Institute of Technology Gandhinagar,78.0,39.0,78.0,78.0,39.0,39.0,2017,2025,39.0,39.0,39.0,False
7,INST1216,S.R.M. Institute of Science and Technology,55.0,22.0,55.0,55.0,22.0,22.0,2017,2025,33.0,33.0,33.0,False
8,INST0069,Andhra University,69.0,41.0,69.0,69.0,41.0,41.0,2017,2025,28.0,28.0,28.0,False
9,INST0563,Indian Institute of Science Education & Resear...,98.0,75.0,98.0,98.0,75.0,75.0,2017,2025,23.0,23.0,23.0,False


## Question 3: Largest rank declines

year,institution_key,institute_name,rank_start,rank_end,rank_start_low,rank_start_high,rank_end_low,rank_end_high,start_year,end_year,rank_change,rank_change_low,rank_change_high,uses_estimated_rank
0,INST1278,Savitribai Phule Pune University,18.0,91.0,18.0,18.0,91.0,91.0,2017,2025,-73.0,-73.0,-73.0,False
1,INST1488,Tamil Nadu Agricultural University,28.0,88.0,28.0,28.0,88.0,88.0,2017,2025,-60.0,-60.0,-60.0,False
2,INST0139,Bharath Institute of Higher Education & Research,35.0,90.0,35.0,35.0,90.0,90.0,2017,2025,-55.0,-55.0,-55.0,False
3,INST1135,Punjab Agricultural University,40.0,81.0,40.0,40.0,81.0,81.0,2017,2025,-41.0,-41.0,-41.0,False
4,INST0675,Jamia Hamdard,42.0,74.0,42.0,42.0,74.0,74.0,2017,2025,-32.0,-32.0,-32.0,False
5,INST0141,Bharathiar University,45.0,76.0,45.0,45.0,76.0,76.0,2017,2025,-31.0,-31.0,-31.0,False
6,INST0565,Indian Institute of Science Education & Resear...,29.0,55.0,29.0,29.0,55.0,55.0,2017,2025,-26.0,-26.0,-26.0,False
7,INST0590,Indian Institute of Technology Ropar,32.0,56.0,32.0,32.0,56.0,56.0,2017,2025,-24.0,-24.0,-24.0,False
8,INST0608,Institute of Chemical Technology,41.0,64.0,41.0,41.0,64.0,64.0,2017,2025,-23.0,-23.0,-23.0,False
9,INST0564,Indian Institute of Science Education & Resear...,44.0,67.0,44.0,44.0,67.0,67.0,2017,2025,-23.0,-23.0,-23.0,False


... (13 outputs left out)

,institution_key,institute_name,rank_volatility,years_ranked,years_in_window,rank_coverage,is_volatility_outlier,rank_basis
0,INST0139,Bharath Institute of Higher Education & Research,0.250,8,9,0.889,False,Exact ranks
1,INST1276,Saveetha Institute of Medical and Technical Sc...,0.242,9,9,1.000,False,Exact ranks
2,INST1278,Savitribai Phule Pune University,0.238,9,9,1.000,False,Exact ranks
3,INST0763,King George's Medical University,0.227,8,9,0.889,False,Exact ranks
4,INST1493,Tata Institute of Social Sciences,0.218,7,9,0.778,False,Exact ranks
5,INST0812,Lovely Professional University,0.218,6,9,0.667,False,Exact ranks
6,INST1394,Sri Ramachandra Institute of Higher Education ...,0.204,6,9,0.667,False,Exact ranks
7,INST0563,Indian Institute of Science Education & Resear...,0.195,9,9,1.000,False,Exact ranks
8,INST0055,Amity University - Gautam Budh Nagar,0.191,9,9,1.000,False,Exact ranks
9,INST0253,Cochin University of Science and Technology,0.190,8,9,0.889,False,Exact ranks


## Endpoint coverage: exact ranks versus rank-band estimates

,comparison,start_year,end_year,institutions_matched,band_midpoint_estimates_used
0,Exact ranks only,2017,2025,63,0
1,Exact ranks + band midpoint estimates,2017,2025,108,45


Dashboard functions are available from engine.analysis. Each question_* function accepts df and keyword filters and returns (result_df, plotly_fig).
